In [ ]:
import torch
import myneural as mn

In [ ]:
class Block(mn.Module):
    def __init__(self, config: mn.Config, dtype = torch.float32, device = 'cpu'):
        self.attention = mn.MultiHeadAttention(config, dtype, device)
        self.feed_forward = mn.FeedForward(config, dtype, device)
        self.ln1 = mn.LayerNorm(config.n_embd, 1e-8, dtype, device)
        self.ln2 = mn.LayerNorm(config.n_embd, 1e-8, dtype, device)
        
    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x

class Transformer(mn.Module):
    def __init__(self, config: mn.Config, dtype = torch.float32, device = 'cpu'):
        self.config = config
        self.token_embedding = mn.Embedding(config.vocab_size, config.n_embd, dtype, device)
        self.pos_embedding = mn.Embedding(config.context_length, config.n_embd, dtype, device)
        self.transformer = mn.Sequential([
            Block(config, dtype, device) for _ in range(config.n_layer)
        ])
        self.ln_final = mn.LayerNorm(config.n_embd, 1e-8, dtype, device)


    def forward(self, idx):
        token_embedding = self.token_embedding(idx) # (B, T) => (B, T, C)
        pos_embedding = self.pos_embedding(torch.arange(0, self.config.context_length, device = idx.device)) # (T,C)
        x = token_embedding + pos_embedding # (B, T, C) + (B_b, T, C)
        x = self.transformer(x)
        logits = x @ self.token_embedding.embedding.T # lm_head, but directly applied 
        return logits